In [1]:
# Importação das bibliotecas principais
# TensorFlow será utilizado para Deep Learning
# TensorFlow Datasets será usado para carregar o dataset Cats vs Dogs
# Matplotlib será usado para visualização dos resultados

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt


In [2]:
# Carregamento do dataset Cats vs Dogs
# O dataset é dividido em 80% para treino e 20% para validação
# as_supervised=True retorna pares (imagem, rótulo)

(train_ds, val_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

# Verificando as informações do dataset
print(metadata)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.X5VKLA_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
tfds.core.DatasetInfo(
    name='cats_vs_dogs',
    full_name='cats_vs_dogs/4.0.1',
    description="""
    A large set of images of cats and dogs. There are 1738 corrupted images that are dropped.
    """,
    homepage='https://www.microsoft.com/en-us/download/details.aspx?id=54765',
    data_dir='/root/tensorflow_datasets/cats_vs_dogs/4.0.1',
    file_format=tfrecord,
    download_size=786.67 MiB,
    dataset_size=1.04 GiB,
    features=FeaturesDict({
        'image': Image(shape=(None, None, 3), dtype=uint8),
        'image/filename': Text(shape=(), dtype=string),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'train': <SplitInfo num_examples=23262, num_shards=16>,
    },
    citation="""@Inproceedings 

In [3]:
# Definição do tamanho das imagens
# MobileNetV2 trabalha bem com imagens 160x160
IMG_SIZE = (160, 160)

# Definição do tamanho do batch
BATCH_SIZE = 32

# Função de pré-processamento
# Redimensiona as imagens
# Normaliza os valores dos pixels para o intervalo [0,1]

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Aplicando o pré-processamento
# batch(): agrupa os dados
# prefetch(): melhora o desempenho durante o treinamento

train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [4]:
# Carregamento do modelo MobileNetV2
# include_top=False remove a camada final original
# weights='imagenet' utiliza pesos pré-treinados

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights='imagenet'
)

# Congelamento das camadas convolucionais
# Isso garante que apenas as camadas finais serão treinadas

base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# Construção do modelo final
# GlobalAveragePooling reduz a dimensionalidade
# Dense adiciona camadas totalmente conectadas
# A última camada usa sigmoid para classificação binária

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Exibindo o resumo do modelo
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# Compilação do modelo
# Adam é um otimizador eficiente
# Binary Crossentropy é usada para classificação binária
# Accuracy será usada como métrica de avaliação

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# Treinamento do modelo
# O modelo será treinado por 5 épocas
# Os dados de validação são usados para avaliar a generalização

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
582/582 ━━━━━━━━━━━━━━━━━━━━ 503s 854ms/step - accuracy: 0.9609 - loss: 0.1021 - val_accuracy: 0.9809 - val_loss: 0.0554
Epoch 2/5
123/582 ━━━━━━━━━━━━━━━━━━━━ 5:08 672ms/step - accuracy: 0.9849 - loss: 0.0440

In [ ]:
# Extração dos valores de acurácia e perda
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Criação dos gráficos
plt.figure(figsize=(12,4))

# Gráfico de acurácia
plt.subplot(1,2,1)
plt.plot(acc, label='Treino')
plt.plot(val_acc, label='Validação')
plt.title('Acurácia')
plt.legend()

# Gráfico de loss
plt.subplot(1,2,2)
plt.plot(loss, label='Treino')
plt.plot(val_loss, label='Validação')
plt.title('Loss')
plt.legend()

plt.show()
